In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_02 — Synthetic Data Generation
# MAGIC **Reads calibration stats from Bronze → generates 50,000 synthetic Indian borrower profiles**
# MAGIC
# MAGIC What this notebook does:
# MAGIC - Reads `xscore.bronze.calibration_stats` (real distributions mined in NB_01)
# MAGIC - Defines 5 India-specific borrower archetypes (SHG woman, gig worker, kirana owner, salaried informal, rural farmer)
# MAGIC - Generates 50,000 unified profiles — each person has ALL 6 XScore pillars in one row
# MAGIC - Generates 24-month bill payment history (Pillar 1 time-series)
# MAGIC - Generates 1-month UPI transaction history (Pillar 2 time-series)
# MAGIC - Computes a realistic default label for each person (the training target)
# MAGIC - Writes everything to `xscore.bronze` as Delta tables
# MAGIC
# MAGIC **Depends on:** NB_01 complete  
# MAGIC **Runtime:** ~5 minutes  
# MAGIC **Next:** NB_03_silver_cleaning

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 1 — Install dependencies and setup

# COMMAND ----------

%pip install faker numpy scipy pandas --quiet

# COMMAND ----------

import numpy as np
import pandas as pd
from scipy.stats import lognorm
from faker import Faker
from pyspark.sql import functions as F
import json

fake = Faker("en_IN")
np.random.seed(42)

spark.sql("USE CATALOG xscore")
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("✓ Libraries loaded")
print("✓ Random seed set to 42 (reproducible generation)")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 2 — Load calibration stats from Bronze

# COMMAND ----------

# Read the real distributions we mined in NB_01
# These numbers make our synthetic data realistic, not random

calib_raw = spark.table("xscore.bronze.calibration_stats").collect()
CALIB = {row["stat_key"]: row["stat_value"] for row in calib_raw}

print("Calibration stats loaded from xscore.bronze.calibration_stats:\n")
for k, v in sorted(CALIB.items()):
    print(f"  {k:<35} {v:>15,.2f}")

# Key values we'll use in generation
HC_DEFAULT_RATE = CALIB.get("hc_default_rate", 0.08)
UPI_AVG_AMOUNT  = CALIB.get("upi_avg_amount",  500.0)
UPI_STD_AMOUNT  = CALIB.get("upi_std_amount",  400.0)

print(f"\nKey calibration values:")
print(f"  Real-world default rate : {HC_DEFAULT_RATE:.2%}")
print(f"  UPI avg transaction     : ₹{UPI_AVG_AMOUNT:,.0f}")
print(f"  UPI std transaction     : ₹{UPI_STD_AMOUNT:,.0f}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 3 — Define the 5 borrower archetypes
# MAGIC
# MAGIC Every number here is grounded in real Indian data:
# MAGIC - Income means/stds from NSSO PLFS 2022-23 published summaries
# MAGIC - Default rates scaled from Home Credit + India NBFC NPA data
# MAGIC - UPI volumes from NPCI monthly statistics
# MAGIC - Bill payment rates estimated from BBPS collection data
# MAGIC - Asset ownership from NABARD NAFIS 2016-17

# COMMAND ----------

ARCHETYPES = {

    "shg_woman": {
        "weight"              : 0.20,
        "n"                   : 10000,
        # Income — NSSO PLFS rural self-employed women
        "income_mean"         : 9100,
        "income_std"          : 2800,
        "income_min"          : 2000,
        # Pillar 1 — bill payment
        "bill_ontime_mean"    : 0.65,
        "bill_ontime_std"     : 0.12,
        # Pillar 2 — UPI
        "upi_txn_min"         : 5,
        "upi_txn_max"         : 20,
        "upi_amount_scale"    : 0.55,   # relative to UPI_AVG_AMOUNT
        # Pillar 3 — assets
        "land_prob"           : 0.35,
        "land_acres_mean"     : 0.8,
        "vehicle_prob"        : 0.18,
        "has_fd_prob"         : 0.08,
        # Pillar 4 — income/employment
        "itr_prob"            : 0.02,
        "gst_prob"            : 0.00,
        "employment_mean_mths": 24,
        # Pillar 5 — identity/govt
        "jan_dhan_prob"       : 0.78,
        "shg_prob"            : 0.92,
        "dbt_prob"            : 0.65,
        "svanidhi_prob"       : 0.05,
        # Pillar 6 — digital stability
        "sim_tenure_min"      : 12,
        "sim_tenure_max"      : 72,
        # Default
        "base_default_rate"   : 0.13,
        # Geography
        "states"              : ["Maharashtra", "Tamil Nadu", "Andhra Pradesh",
                                 "Karnataka", "West Bengal", "Rajasthan"],
    },

    "gig_worker": {
        "weight"              : 0.25,
        "n"                   : 12500,
        "income_mean"         : 18000,
        "income_std"          : 5500,
        "income_min"          : 8000,
        "bill_ontime_mean"    : 0.74,
        "bill_ontime_std"     : 0.10,
        "upi_txn_min"         : 30,
        "upi_txn_max"         : 90,
        "upi_amount_scale"    : 0.90,
        "land_prob"           : 0.04,
        "land_acres_mean"     : 0.0,
        "vehicle_prob"        : 0.72,
        "has_fd_prob"         : 0.10,
        "itr_prob"            : 0.11,
        "gst_prob"            : 0.00,
        "employment_mean_mths": 18,
        "jan_dhan_prob"       : 0.42,
        "shg_prob"            : 0.02,
        "dbt_prob"            : 0.25,
        "svanidhi_prob"       : 0.01,
        "sim_tenure_min"      : 6,
        "sim_tenure_max"      : 60,
        "base_default_rate"   : 0.09,
        "states"              : ["Delhi", "Maharashtra", "Karnataka",
                                 "Telangana", "Tamil Nadu", "Gujarat"],
    },

    "kirana_owner": {
        "weight"              : 0.20,
        "n"                   : 10000,
        "income_mean"         : 27000,
        "income_std"          : 9000,
        "income_min"          : 10000,
        "bill_ontime_mean"    : 0.80,
        "bill_ontime_std"     : 0.09,
        "upi_txn_min"         : 80,
        "upi_txn_max"         : 300,
        "upi_amount_scale"    : 0.70,
        "land_prob"           : 0.42,
        "land_acres_mean"     : 0.3,
        "vehicle_prob"        : 0.58,
        "has_fd_prob"         : 0.25,
        "itr_prob"            : 0.38,
        "gst_prob"            : 0.45,
        "employment_mean_mths": 60,
        "jan_dhan_prob"       : 0.28,
        "shg_prob"            : 0.05,
        "dbt_prob"            : 0.15,
        "svanidhi_prob"       : 0.00,
        "sim_tenure_min"      : 24,
        "sim_tenure_max"      : 96,
        "base_default_rate"   : 0.07,
        "states"              : ["Uttar Pradesh", "Maharashtra", "Rajasthan",
                                 "Gujarat", "West Bengal", "Madhya Pradesh"],
    },

    "salaried_informal": {
        "weight"              : 0.20,
        "n"                   : 10000,
        "income_mean"         : 22000,
        "income_std"          : 5000,
        "income_min"          : 10000,
        "bill_ontime_mean"    : 0.83,
        "bill_ontime_std"     : 0.08,
        "upi_txn_min"         : 18,
        "upi_txn_max"         : 55,
        "upi_amount_scale"    : 1.10,
        "land_prob"           : 0.09,
        "land_acres_mean"     : 0.0,
        "vehicle_prob"        : 0.52,
        "has_fd_prob"         : 0.22,
        "itr_prob"            : 0.26,
        "gst_prob"            : 0.00,
        "employment_mean_mths": 48,
        "jan_dhan_prob"       : 0.22,
        "shg_prob"            : 0.01,
        "dbt_prob"            : 0.12,
        "svanidhi_prob"       : 0.00,
        "sim_tenure_min"      : 12,
        "sim_tenure_max"      : 84,
        "base_default_rate"   : 0.06,
        "states"              : ["Delhi", "Maharashtra", "Karnataka",
                                 "Tamil Nadu", "Gujarat", "Haryana"],
    },

    "rural_farmer": {
        "weight"              : 0.15,
        "n"                   : 7500,
        "income_mean"         : 11000,
        "income_std"          : 5500,
        "income_min"          : 2000,
        "bill_ontime_mean"    : 0.55,
        "bill_ontime_std"     : 0.15,
        "upi_txn_min"         : 3,
        "upi_txn_max"         : 18,
        "upi_amount_scale"    : 0.50,
        "land_prob"           : 0.78,
        "land_acres_mean"     : 2.5,
        "vehicle_prob"        : 0.28,
        "has_fd_prob"         : 0.06,
        "itr_prob"            : 0.04,
        "gst_prob"            : 0.00,
        "employment_mean_mths": 36,
        "jan_dhan_prob"       : 0.82,
        "shg_prob"            : 0.28,
        "dbt_prob"            : 0.70,
        "svanidhi_prob"       : 0.00,
        "sim_tenure_min"      : 6,
        "sim_tenure_max"      : 60,
        "base_default_rate"   : 0.15,
        "states"              : ["Uttar Pradesh", "Madhya Pradesh", "Bihar",
                                 "Rajasthan", "Maharashtra", "Punjab"],
    },
}

TOTAL = sum(a["n"] for a in ARCHETYPES.values())
print(f"Total profiles to generate: {TOTAL:,}\n")
for seg, arch in ARCHETYPES.items():
    print(f"  {seg:<22} {arch['n']:>6,}  ({arch['weight']:.0%})  "
          f"base_default={arch['base_default_rate']:.0%}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 4 — Profile generator function

# COMMAND ----------

def generate_profile(user_id: str, segment: str, arch: dict) -> dict:
    """
    Generate one complete synthetic borrower.
    All 6 XScore pillars in a single dict.
    Everything is correlated through income_factor.
    """
    a = arch

    # ── ROOT: income (everything correlates to this) ─────────
    income = float(max(
        a["income_min"],
        np.random.normal(a["income_mean"], a["income_std"])
    ))
    # income_factor: 1.0 = average for this segment
    # < 1.0 = poorer than average → worse bill payment, higher default
    # > 1.0 = richer than average → better behaviour, lower default
    income_factor = income / a["income_mean"]

    # ── PILLAR 1: Bill payment ────────────────────────────────
    raw_ontime = np.random.normal(a["bill_ontime_mean"], a["bill_ontime_std"])
    bill_ontime_rate = float(np.clip(
        raw_ontime * (0.80 + 0.20 * income_factor), 0.10, 0.99
    ))
    # Streak = longest run of consecutive months with no missed bills
    # Geometric distribution: higher on-time rate → longer streak
    bill_streak = int(min(24, np.random.geometric(
        p=max(0.05, 1.0 - bill_ontime_rate)
    )))
    nach_bounce_rate = float(np.clip(
        np.random.normal(0.08 * (1.5 - income_factor), 0.04), 0.0, 0.45
    ))

    # ── PILLAR 2: UPI & digital flow ─────────────────────────
    upi_txn_per_month = int(np.random.uniform(
        a["upi_txn_min"], a["upi_txn_max"]
    ))
    upi_avg_amount = float(max(50.0,
        np.random.normal(
            UPI_AVG_AMOUNT * a["upi_amount_scale"] * income_factor,
            UPI_STD_AMOUNT * 0.4
        )
    ))
    monthly_credit = income * float(np.random.uniform(0.88, 1.12))
    savings_ratio  = float(np.clip(
        np.random.normal(0.06 * income_factor, 0.08), -0.20, 0.45
    ))
    upi_failure_rate = float(np.clip(
        np.random.normal(0.04 * (1.5 - income_factor), 0.02), 0.0, 0.30
    ))
    merchant_entropy = float(np.clip(
        np.random.normal(
            2.5 if segment == "kirana_owner" else 1.8 * income_factor,
            0.4
        ), 0.5, 4.0
    ))

    # ── PILLAR 3: Assets & property ──────────────────────────
    owns_land    = bool(np.random.random() < a["land_prob"])
    land_acres   = float(
        lognorm.rvs(s=0.9, scale=max(0.1, a["land_acres_mean"]))
        if (owns_land and a["land_acres_mean"] > 0) else 0.0
    )
    owns_vehicle = bool(np.random.random() <
                        min(0.95, a["vehicle_prob"] * (0.7 + 0.3 * income_factor)))
    prop_tax_compliant = bool(np.random.random() < bill_ontime_rate) if owns_land else False
    bank_vintage_months = int(max(3, np.random.normal(36 * income_factor, 18)))
    has_fd_or_rd = bool(np.random.random() < a["has_fd_prob"] * income_factor)

    # ── PILLAR 4: Income & employment ────────────────────────
    itr_filed   = bool(np.random.random() < a["itr_prob"])
    gst_reg     = bool(np.random.random() < a["gst_prob"] * income_factor)
    emp_months  = int(max(1, np.random.exponential(a["employment_mean_mths"])))
    # Income declared: ITR filers declare more honestly
    declared_ratio = float(np.random.uniform(0.85, 1.0) if itr_filed
                           else np.random.uniform(0.55, 0.95))

    # ── PILLAR 5: Identity & govt signals ────────────────────
    jan_dhan   = bool(np.random.random() < a["jan_dhan_prob"])
    shg_member = bool(np.random.random() < a["shg_prob"])
    shg_months = int(np.random.uniform(6, 60)) if shg_member else 0
    dbt_active = bool(np.random.random() < a["dbt_prob"]) if jan_dhan else False
    dbt_months = int(np.random.uniform(3, 24)) if dbt_active else 0
    svanidhi   = bool(np.random.random() < a["svanidhi_prob"])
    scheme_count = int(np.random.poisson(1.5 if jan_dhan else 0.3))

    # ── PILLAR 6: Digital stability ───────────────────────────
    sim_tenure      = int(np.random.uniform(a["sim_tenure_min"], a["sim_tenure_max"]))
    location_stable = float(np.clip(
        np.random.normal(0.75 + 0.15 * income_factor, 0.15), 0.1, 1.0
    ))
    kyc_months_old  = int(np.random.uniform(0, 48))
    fraud_flag      = bool(np.random.random() < 0.008)

    # ── DEFAULT LABEL (the training target) ──────────────────
    # This formula encodes domain knowledge about what drives default.
    # Each factor modifies the segment's base default probability.
    p = a["base_default_rate"]
    p *= max(0.30, 1.80 - 0.80 * income_factor)      # income ↑ → risk ↓
    p *= max(0.50, 1.60 - 0.60 * bill_ontime_rate)   # payment discipline ↑ → risk ↓
    p *= (0.55 if itr_filed else 1.00)                # ITR = formal = lower risk
    p *= (0.65 if owns_land else 1.00)                # land = collateral
    p *= (0.70 if savings_ratio > 0.05 else 1.25)    # saving habit ↑ → risk ↓
    p *= (0.80 if svanidhi else 1.00)                 # small loan repaid = proven
    p *= (1.80 if fraud_flag else 1.00)               # fraud = very high risk
    p *= max(0.60, 1.30 - 0.01 * bank_vintage_months) # banking tenure ↑ → risk ↓
    p *= (0.75 if shg_member else 1.00)               # SHG member = group accountability
    default_prob  = float(np.clip(p, 0.005, 0.65))
    default_label = int(np.random.random() < default_prob)

    state = str(np.random.choice(a["states"]))

    return {
        # Identity
        "user_id"               : user_id,
        "segment"               : segment,
        "state"                 : state,
        # Pillar 1 — bill payment
        "bill_ontime_rate_24m"  : round(bill_ontime_rate, 4),
        "bill_streak_months"    : bill_streak,
        "nach_bounce_rate"      : round(nach_bounce_rate, 4),
        # Pillar 2 — UPI & digital
        "income_monthly"        : round(income, 2),
        "upi_txn_per_month"     : upi_txn_per_month,
        "upi_avg_txn_amount"    : round(upi_avg_amount, 2),
        "upi_monthly_credit"    : round(monthly_credit, 2),
        "savings_ratio"         : round(savings_ratio, 4),
        "upi_failure_rate"      : round(upi_failure_rate, 4),
        "merchant_entropy"      : round(merchant_entropy, 4),
        # Pillar 3 — assets
        "owns_land"             : int(owns_land),
        "land_acres"            : round(land_acres, 3),
        "owns_vehicle"          : int(owns_vehicle),
        "prop_tax_compliant"    : int(prop_tax_compliant),
        "bank_vintage_months"   : bank_vintage_months,
        "has_fd_or_rd"          : int(has_fd_or_rd),
        # Pillar 4 — income/employment
        "itr_filed"             : int(itr_filed),
        "gst_registered"        : int(gst_reg),
        "employment_months"     : emp_months,
        "income_declared_ratio" : round(declared_ratio, 3),
        # Pillar 5 — identity/govt
        "jan_dhan_active"       : int(jan_dhan),
        "shg_member"            : int(shg_member),
        "shg_months"            : shg_months,
        "dbt_months"            : dbt_months,
        "svanidhi_repaid"       : int(svanidhi),
        "govt_scheme_count"     : scheme_count,
        # Pillar 6 — digital stability
        "sim_tenure_months"     : sim_tenure,
        "location_stability"    : round(location_stable, 4),
        "kyc_months_old"        : kyc_months_old,
        "fraud_flag"            : int(fraud_flag),
        # Target
        "default_probability"   : round(default_prob, 5),
        "default_label"         : default_label,
    }

print("✓ Profile generator function defined")
print(f"  Each profile has {len(generate_profile('test', 'gig_worker', ARCHETYPES['gig_worker']))} features")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 5 — Generate all 50,000 profiles

# COMMAND ----------

from datetime import datetime

print(f"Generating {TOTAL:,} synthetic profiles...")
print(f"Started: {datetime.now().strftime('%H:%M:%S')}\n")

all_profiles = []
uid = 1

for seg_name, arch in ARCHETYPES.items():
    n = arch["n"]
    print(f"  Generating {n:,} {seg_name} profiles...", end=" ")
    for _ in range(n):
        profile = generate_profile(f"USR{uid:06d}", seg_name, arch)
        all_profiles.append(profile)
        uid += 1
    print("done")

profiles_pd = pd.DataFrame(all_profiles)

print(f"\nFinished: {datetime.now().strftime('%H:%M:%S')}")
print(f"Total profiles: {len(profiles_pd):,}")
print(f"\nDefault rate by segment:")
print(
    profiles_pd.groupby("segment")["default_label"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "default_rate", "sum": "defaults", "count": "total"})
    .round(3)
    .to_string()
)

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 6 — Generate 24-month bill payment history

# COMMAND ----------

print("Generating bill payment time-series (24 months × 3-4 bills per user)...")
print(f"Expected rows: ~{TOTAL * 3.5 * 24 / 1e6:.1f}M\n")

BILL_TYPES = {
    "electricity": {"mean": 820,  "std": 320, "all_segments": True},
    "mobile"     : {"mean": 299,  "std": 100, "all_segments": True},
    "water"      : {"mean": 175,  "std": 65,  "all_segments": False},
    "dth"        : {"mean": 249,  "std": 150, "all_segments": False},
}

bill_records = []
for _, row in profiles_pd.iterrows():
    uid          = row["user_id"]
    ontime_rate  = row["bill_ontime_rate_24m"]
    income       = row["income_monthly"]
    segment      = row["segment"]

    for month in range(1, 25):
        for btype, binfo in BILL_TYPES.items():
            # Gig workers typically don't have separate water bills (rented rooms)
            if not binfo["all_segments"] and segment == "gig_worker":
                continue
            # DTH is optional — not everyone has it
            if btype == "dth" and np.random.random() < 0.35:
                continue

            amount = float(max(30.0, np.random.normal(
                binfo["mean"] * (0.80 + 0.20 * income / 20000),
                binfo["std"]
            )))
            paid_on_time = bool(np.random.random() < ontime_rate)
            # Late payment: exponential distribution — most are slightly late,
            # few are very late (cash-crunch months)
            days_late = 0 if paid_on_time else int(np.random.exponential(10))

            bill_records.append({
                "user_id"     : uid,
                "month"       : month,
                "bill_type"   : btype,
                "amount"      : round(amount, 2),
                "paid_on_time": int(paid_on_time),
                "days_late"   : min(days_late, 90),
            })

bills_pd = pd.DataFrame(bill_records)
print(f"✓ Generated {len(bills_pd):,} bill records")
print(f"  Overall on-time rate: {bills_pd['paid_on_time'].mean():.1%}")
print(f"  Avg days late (when late): {bills_pd[bills_pd['days_late']>0]['days_late'].mean():.1f} days")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 7 — Generate 1-month UPI transaction history

# COMMAND ----------

print("Generating UPI transaction time-series (1 month)...")
print(f"Expected rows: ~{TOTAL * 40 * 1 / 1e6:.1f}M (varies by segment)\n")

MERCHANT_CATEGORIES = [
    ("grocery",       0.22),
    ("fuel",          0.08),
    ("medical",       0.06),
    ("restaurant",    0.10),
    ("utility_pay",   0.07),
    ("school_fees",   0.04),
    ("clothing",      0.06),
    ("electronics",   0.03),
    ("transport",     0.09),
    ("p2p_transfer",  0.15),
    ("rent_payment",  0.05),
    ("other",         0.05),
]
CATS, CAT_PROBS = zip(*MERCHANT_CATEGORIES)

# Real Indian bank UPI codes
BANK_CODES = [
    "@okaxis", "@oksbi", "@okhdfcbank", "@okicici",
    "@ybl", "@ibl", "@paytm", "@apl", "@barodampay", "@upi"
]

upi_records = []
for _, row in profiles_pd.iterrows():
    uid        = row["user_id"]
    txn_count  = row["upi_txn_per_month"]
    avg_amt    = row["upi_avg_txn_amount"]
    fail_rate  = row["upi_failure_rate"]

    for month in range(1, 2):  # Changed from range(1, 4) to range(1, 2) - just 1 month
        # Add some month-to-month variance in transaction count
        n_txns = max(1, int(np.random.normal(txn_count, txn_count * 0.20)))
        for _ in range(n_txns):
            # Exponential distribution for amounts — many small, few large
            amount   = float(max(10.0, np.random.exponential(avg_amt)))
            category = str(np.random.choice(CATS, p=CAT_PROBS))
            status   = "FAILED" if np.random.random() < fail_rate else "SUCCESS"

            upi_records.append({
                "user_id"  : uid,
                "month"    : month,
                "amount"   : round(amount, 2),
                "category" : category,
                "bank_code": str(np.random.choice(BANK_CODES)),
                "status"   : status,
            })

upi_txns_pd = pd.DataFrame(upi_records)
print(f"✓ Generated {len(upi_txns_pd):,} UPI transaction records")
print(f"  Overall failure rate: {(upi_txns_pd['status']=='FAILED').mean():.1%}")
print(f"  Avg transaction amount: ₹{upi_txns_pd['amount'].mean():,.0f}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 8 — Write all three tables to xscore.bronze

# COMMAND ----------

print("Writing to Delta tables...\n")

# ── 1. User profiles ─────────────────────────────────────────
profiles_spark = spark.createDataFrame(profiles_pd)
(profiles_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("segment")
    .saveAsTable("xscore.bronze.synthetic_profiles"))

print(f"✓ xscore.bronze.synthetic_profiles written")
print(f"  {len(profiles_pd):,} rows\n")

# ── 2. Bill payment history ──────────────────────────────────
bills_spark = spark.createDataFrame(bills_pd)
(bills_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("xscore.bronze.synthetic_bills"))

print(f"✓ xscore.bronze.synthetic_bills written")
print(f"  {len(bills_pd):,} rows\n")

# ── 3. UPI transaction history ───────────────────────────────
upi_spark = spark.createDataFrame(upi_txns_pd)
(upi_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("xscore.bronze.synthetic_upi_txns"))

print(f"✓ xscore.bronze.synthetic_upi_txns written")
print(f"  {len(upi_txns_pd):,} rows\n")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 9 — Final summary

# COMMAND ----------

print("=" * 60)
print("  NB_02 SYNTHETIC DATA GENERATION — COMPLETE")
print("=" * 60)

print("\nTables created in xscore.bronze:\n")
print(f"  1. synthetic_profiles      {len(profiles_pd):>10,} rows")
print(f"     ↳ 50K borrowers × 35 features across 6 XScore pillars\n")

print(f"  2. synthetic_bills         {len(bills_pd):>10,} rows")
print(f"     ↳ 24-month bill payment history for Pillar 1\n")

print(f"  3. synthetic_upi_txns      {len(upi_txns_pd):>10,} rows")
print(f"     ↳ 1-month UPI transaction history for Pillar 2\n")

print("=" * 60)
print("\n  Overall default rate: "
      f"{profiles_pd['default_label'].mean():.2%}")
print(f"  (target: {HC_DEFAULT_RATE:.2%} from Home Credit real data)")

print("\n  NEXT: Run NB_03_silver_cleaning to clean & enrich")
print("  (feature engineering, outlier handling, schema alignment)")
print("=" * 60)